# Thesis v19 AWG Basic Usage

This notebook shows the minimal workflow for using the `thesis_v19` overlay: generate two `int16` waveforms, preview them, load them into the DAC BRAM players, and enable or disable the RF outputs.

## Imports

The path setup keeps the notebook runnable both from the repository root and from inside the `notebooks/` directory.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np

from firmware import OverlayController
from lib.signals import normalize_to_int16, sawtooth, sine
from lib.plotting import plot_fft, plot_time

## AWG Parameters

The BRAM players in `thesis_v19` hold 131072 signed 16-bit samples per DAC. The RFDC configuration targets 9.8304 GSPS on DAC0 and DAC2.

In [ ]:
DAC_SAMPLE_RATE = 9.8304e9
BUFFER_LEN = 131072
DAC_PEAK = 0.8 * np.iinfo(np.int16).max

## Generate Waveforms

The overlay controller expects `int16` samples. The helper functions generate floating-point waveforms first and then scale them to the DAC range.

In [ ]:
dac0_waveform = normalize_to_int16(
    sine(freq_hz=100e6, sample_rate=DAC_SAMPLE_RATE, num_samples=BUFFER_LEN),
    peak=DAC_PEAK,
)

dac2_waveform = normalize_to_int16(
    sawtooth(freq_hz=50e6, sample_rate=DAC_SAMPLE_RATE, num_samples=BUFFER_LEN),
    peak=DAC_PEAK,
)

dac0_waveform.dtype, dac0_waveform.shape, dac2_waveform.dtype, dac2_waveform.shape

## Preview DAC0

In [ ]:
plot_time(dac0_waveform, DAC_SAMPLE_RATE, samples=2048, title="DAC0 waveform");
plot_fft(dac0_waveform, DAC_SAMPLE_RATE, title="DAC0 spectrum");

## Preview DAC2

In [ ]:
plot_time(dac2_waveform, DAC_SAMPLE_RATE, samples=2048, title="DAC2 waveform");
plot_fft(dac2_waveform, DAC_SAMPLE_RATE, title="DAC2 spectrum");

## Load the Overlay

Instantiating `OverlayController` configures the RFSoC clocks, downloads the bitstream, binds `dac0` and `dac2`, and disables both DAC players initially.

In [ ]:
ol = OverlayController()
ol.info()

## Program the DAC Players

`load_waveform()` writes the BRAM and sets the hardware waveform length. If the DAC was already enabled, it is disabled during the write and restored afterwards.

In [ ]:
ol.dac0.load_waveform(dac0_waveform)
ol.dac2.load_waveform(dac2_waveform)

ol.info()

## Enable Outputs

In [ ]:
ol.dac0.enable()
ol.dac2.enable()

ol.dac0.is_enabled(), ol.dac2.is_enabled()

## Disable Outputs

Run this cell when you are finished or before changing cabling/instrument settings.

In [ ]:
ol.dac0.disable()
ol.dac2.disable()

ol.info()